cocoon_wav2lip_batch_pipeline_colab_LIPSYNC_FIXED.ipynb

Automatically generated by Colab.

Original file is located at
    https://colab.research.google.com/drive/1S1-tcYGB5gw9WCTdK2cqdAaJRPEZ2NVd

# Cocoon Batch Video Pipeline — Wav2Lip + FFmpeg, Lip-Sync Fixed

Notebook này dùng lại cấu trúc Drive cũ:

```text
/content/drive/MyDrive/cocoon_ai_video/
  data/
    master_script.json
    outputs/
      cocoon_test_001_tts_package/
        audio/
          S001.wav
          S002.wav
          ...
    video_templates/
      S001.mp4
      S002.mp4
      ...
      HOST_TALK.mp4
      CTA.mp4
      FAQ_ANSWER.mp4
      HOST_PHONE_READING.mp4
      PRODUCT_CLOSEUP.mp4
    scene_images/                 # optional cho scene không cần lip-sync
  models/
    Wav2Lip-SD-GAN.pt
```

Bản này sửa các lỗi chính:

1. Patch `torch.load(..., weights_only=False)` cho PyTorch 2.6+.
2. Patch `load_model()` để hỗ trợ cả checkpoint dạng `state_dict` và TorchScript module.
3. Không tạo final từ `*_base.mp4` kiểu mux-only cho scene cần lip-sync.
4. Với `needs_lipsync=True`, bắt buộc chạy Wav2Lip thành công. Nếu fail thì dừng, không sinh video lệch miệng.
5. Copy input từ Google Drive sang `/content/wav2lip_runtime` trước khi chạy Wav2Lip.
6. Sau Wav2Lip, ép mux lại audio gốc vào video output để đảm bảo có tiếng.
7. Dùng `_work` thống nhất, không bị lẫn `work` và `_work`.

Input mới chỉ cần thay lại file `.wav` trong:

```text
/content/drive/MyDrive/cocoon_ai_video/data/outputs/cocoon_test_001_tts_package/audio/
```

rồi Run All lại notebook này.

In [ ]:
# ===== FIX VIETNAMESE FONT =====
# Noto Sans hỗ trợ tiếng Việt tốt, tránh lỗi mất dấu / ô vuông khi burn text bằng FFmpeg drawtext.
!apt-get update -y >/dev/null
!apt-get install -y fonts-noto-core fonts-noto-extra fonts-dejavu-core >/dev/null
!fc-cache -fv >/dev/null

# Check font file
!find /usr/share/fonts -iname "*NotoSans*Regular*.ttf" | head -20

## 1. Mount Drive + kiểm tra GPU / FFmpeg

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

!nvidia-smi
!ffmpeg -version | head -n 3

## 2. Cấu hình path

In [ ]:
from pathlib import Path
import os
import json
import shutil
import subprocess
import csv
import textwrap
from datetime import datetime, UTC

# ===== ROOT CONFIG =====
PROJECT_DIR = Path("/content/drive/MyDrive/cocoon_ai_video")

DATA_DIR = PROJECT_DIR / "data"
MODEL_DIR = PROJECT_DIR / "models"

SCRIPT_PATH = DATA_DIR / "master_script.json"

# Audio mới cần đặt đúng folder này, file tên theo scene_id: S001.wav, S002.wav, ...
AUDIO_DIR = DATA_DIR / "outputs" / "cocoon_test_001_tts_package" / "audio"

# Video / ảnh đầu vào
VIDEO_TEMPLATE_DIR = DATA_DIR / "video_templates"
SCENE_IMAGE_DIR = DATA_DIR / "scene_images"

# Wav2Lip
WAV2LIP_DIR = Path("/content/Wav2Lip")
CHECKPOINT_SRC = MODEL_DIR / "Wav2Lip-SD-GAN.pt"
CHECKPOINT_DST = WAV2LIP_DIR / "checkpoints" / "Wav2Lip-SD-GAN.pt"

# Output
JOB_ID_FALLBACK = "cocoon_test_001"
OUTPUT_ROOT = DATA_DIR / "outputs" / f"{JOB_ID_FALLBACK}_video"
WORK_DIR = OUTPUT_ROOT / "_work"
SCENE_OUT_DIR = OUTPUT_ROOT / "scenes"
FINAL_OUT_DIR = OUTPUT_ROOT / "final"
REPORT_DIR = OUTPUT_ROOT / "reports"
RUNTIME_DIR = Path("/content/wav2lip_runtime")

# Render config
TARGET_W = 1080
TARGET_H = 1920
FPS = 25
CRF = 18
PRESET = "veryfast"

# Wav2Lip config
# resize_factor=1 giúp giữ mặt đủ rõ, tránh face detection fail / chất lượng miệng kém.
PADS = ["0", "20", "0", "0"]
RESIZE_FACTOR = "1"
WAV2LIP_BATCH_SIZE ="4"

# BẮT BUỘC lipsync cho scene needs_lipsync=True.
# Nếu Wav2Lip lỗi, notebook dừng thay vì tạo video mux-only lệch miệng.
STRICT_LIPSYNC = True

# ===== RESUME / RERUN CONFIG =====
# False để KHÔNG xóa output cũ. Notebook sẽ chạy tiếp các scene còn thiếu.
# Chỉ bật True khi muốn render lại sạch toàn bộ từ đầu.
CLEAN_OUTPUT = False

# Nếu True: bỏ qua cơ chế resume và render lại tất cả scene.
FORCE_REGENERATE_SCENES = False

# Nếu True: scene nào đã có file output hợp lệ thì skip, chỉ chạy tiếp scene chưa có.
RENDER_ONLY_MISSING_SCENES = True

# Nếu True: sau khi scene đủ, luôn ghép lại playlist/final để cập nhật final video.
REBUILD_FINAL_VIDEO = True

# Tắt test Wav2Lip để tránh chạy lại scene test khi chỉ muốn resume.
# Bật True khi vừa đổi checkpoint/model hoặc muốn debug lipsync.
RUN_WAV2LIP_SMOKE_TEST = False

# Hiệu ứng video cơ học nhẹ bằng FFmpeg: fade, panel text, line trang trí.
ENABLE_MECHANICAL_EDIT_EFFECTS = True

# Cho phép dùng ảnh tĩnh để lipsync nếu thiếu video host.
# Khuyến nghị False vì ảnh tĩnh cho motion kém hơn video mẫu.
ALLOW_STATIC_IMAGE_FOR_LIPSYNC = False

for p in [DATA_DIR, MODEL_DIR, VIDEO_TEMPLATE_DIR, SCENE_IMAGE_DIR, RUNTIME_DIR]:
    p.mkdir(parents=True, exist_ok=True)

print("PROJECT_DIR:", PROJECT_DIR)
print("SCRIPT_PATH:", SCRIPT_PATH)
print("AUDIO_DIR:", AUDIO_DIR)
print("VIDEO_TEMPLATE_DIR:", VIDEO_TEMPLATE_DIR)
print("SCENE_IMAGE_DIR:", SCENE_IMAGE_DIR)
print("CHECKPOINT_SRC:", CHECKPOINT_SRC)

# Font hỗ trợ tiếng Việt tốt hơn DejaVuSans
FONT_PATH = "/usr/share/fonts/truetype/noto/NotoSans-Regular.ttf"

# fallback nếu path khác
from pathlib import Path

def pick_font_path():
    priority_names = [
        "NotoSans-Regular.ttf",
        "NotoSansDisplay-Regular.ttf",
        "DejaVuSans.ttf",
    ]

    for name in priority_names:
        candidates = list(Path("/usr/share/fonts").rglob(name))
        if candidates:
            return str(candidates[0])

    return "/usr/share/fonts/truetype/dejavu/DejaVuSans.ttf"

if not Path(FONT_PATH).exists():
    FONT_PATH = pick_font_path()

# Ép môi trường UTF-8 cho subprocess/ffmpeg.
os.environ["LANG"] = "C.UTF-8"
os.environ["LC_ALL"] = "C.UTF-8"

print("Using font:", FONT_PATH)

## 3. Setup Wav2Lip + patch lỗi PyTorch 2.6 checkpoint

In [ ]:
# Clone Wav2Lip nếu chưa có
if not WAV2LIP_DIR.exists():
    !git clone https://github.com/Rudrabha/Wav2Lip.git /content/Wav2Lip
else:
    print("Wav2Lip already exists:", WAV2LIP_DIR)

# Cài dependency
# Giữ theo hướng setup Colab cũ, thêm numpy<2 để tránh xung đột thư viện legacy.
!pip uninstall -y tensorflow tensorflow-gpu >/dev/null 2>&1 || true
!cd /content/Wav2Lip && pip install -q -r requirements.txt
!pip uninstall -y librosa >/dev/null 2>&1 || true
!pip install -q "numpy<2.0" librosa==0.9.2 opencv-python-headless tqdm scipy

# Tải face detector
s3fd_path = WAV2LIP_DIR / "face_detection" / "detection" / "sfd" / "s3fd.pth"
s3fd_path.parent.mkdir(parents=True, exist_ok=True)
if not s3fd_path.exists():
    !wget -q "https://www.adrianbulat.com/downloads/python-fan/s3fd-619a316812.pth" -O "/content/Wav2Lip/face_detection/detection/sfd/s3fd.pth"
else:
    print("s3fd already exists:", s3fd_path)

# Copy checkpoint từ Drive
CHECKPOINT_DST.parent.mkdir(parents=True, exist_ok=True)
if CHECKPOINT_SRC.exists():
    shutil.copy2(CHECKPOINT_SRC, CHECKPOINT_DST)
    print("Copied checkpoint:", CHECKPOINT_DST)
elif CHECKPOINT_DST.exists():
    print("Checkpoint already exists:", CHECKPOINT_DST)
else:
    raise FileNotFoundError(
        f"Không thấy checkpoint. Đặt file tại {CHECKPOINT_SRC} "
        "hoặc sửa CHECKPOINT_SRC trong cell cấu hình."
    )

print("Setup done.")

from pathlib import Path
import re

inference_path = Path("/content/Wav2Lip/inference.py")

text = inference_path.read_text(encoding="utf-8")

# Backup file gốc 1 lần
backup_path = Path("/content/Wav2Lip/inference.py.original.bak")
if not backup_path.exists():
    backup_path.write_text(text, encoding="utf-8")

# FIX QUAN TRỌNG:
# Patch cả _load() + load_model().
# Lý do: file Wav2Lip-SD-GAN.pt của project này có thể là TorchScript archive,
# không phải checkpoint dict có key "state_dict" như Wav2Lip gốc.
# Nếu chỉ patch torch.load(weights_only=False) mà giữ load_model gốc,
# code sẽ vỡ ở checkpoint["state_dict"].
patched_load_block = """def _load(checkpoint_path):
    # Hỗ trợ checkpoint dạng TorchScript archive trước.
    # Với checkpoint state_dict thường, torch.jit.load sẽ fail và fallback sang torch.load.
    try:
        checkpoint = torch.jit.load(checkpoint_path, map_location=device)
        print("Loaded TorchScript checkpoint.")
        return checkpoint
    except Exception:
        pass

    if device == 'cuda':
        checkpoint = torch.load(checkpoint_path, weights_only=False)
    else:
        checkpoint = torch.load(
            checkpoint_path,
            map_location=lambda storage, loc: storage,
            weights_only=False
        )
    return checkpoint


def load_model(path):
    print("Load checkpoint from: {}".format(path))
    checkpoint = _load(path)

    # Case 1: TorchScript / nn.Module checkpoint.
    # Không được truy cập checkpoint["state_dict"] trong case này.
    if hasattr(checkpoint, "forward") and not isinstance(checkpoint, dict):
        model = checkpoint
        model = model.to(device)
        return model.eval()

    # Case 2: Standard Wav2Lip checkpoint dict.
    model = Wav2Lip()
    if isinstance(checkpoint, dict) and "state_dict" in checkpoint:
        s = checkpoint["state_dict"]
    elif isinstance(checkpoint, dict):
        s = checkpoint
    else:
        raise TypeError("Unsupported checkpoint type: {}".format(type(checkpoint)))

    new_s = {}
    for k, v in s.items():
        new_s[k.replace("module.", "")] = v
    model.load_state_dict(new_s)

    model = model.to(device)
    return model.eval()


def main():"""

pattern = re.compile(
    r"def _load\(checkpoint_path\):.*?def main\(\):",
    flags=re.DOTALL
)

if not pattern.search(text):
    raise RuntimeError("Không tìm thấy block _load/load_model/main trong /content/Wav2Lip/inference.py")

text = pattern.sub(patched_load_block, text)
inference_path.write_text(text, encoding="utf-8")

print("Patched Wav2Lip inference.py with TorchScript + state_dict support")
!grep -n "Loaded TorchScript\|def _load\|torch.load\|def load_model\|state_dict" /content/Wav2Lip/inference.py | head -60

## 4. Load JSON + validate assets

In [ ]:
def load_script(path: Path):
    if not path.exists():
        raise FileNotFoundError(f"Không thấy master_script.json tại: {path}")
    with open(path, "r", encoding="utf-8") as f:
        data = json.load(f)
    if "scenes" not in data:
        raise ValueError("JSON thiếu field 'scenes'.")
    if "playlist" not in data:
        raise ValueError("JSON thiếu field 'playlist'.")
    return data

script = load_script(SCRIPT_PATH)
JOB_ID = script.get("job_id", JOB_ID_FALLBACK)

# Update output theo job_id thật
OUTPUT_ROOT = DATA_DIR / "outputs" / f"{JOB_ID}_video"
WORK_DIR = OUTPUT_ROOT / "_work"
SCENE_OUT_DIR = OUTPUT_ROOT / "scenes"
FINAL_OUT_DIR = OUTPUT_ROOT / "final"
REPORT_DIR = OUTPUT_ROOT / "reports"

if CLEAN_OUTPUT and OUTPUT_ROOT.exists():
    print("Cleaning old output:", OUTPUT_ROOT)
    shutil.rmtree(OUTPUT_ROOT)
else:
    print("Resume mode: giữ output cũ nếu đã có:", OUTPUT_ROOT)

for p in [WORK_DIR, SCENE_OUT_DIR, FINAL_OUT_DIR, REPORT_DIR, RUNTIME_DIR]:
    p.mkdir(parents=True, exist_ok=True)

# Clean runtime cache
for p in RUNTIME_DIR.glob("*"):
    if p.is_file():
        p.unlink()
    elif p.is_dir():
        shutil.rmtree(p)

scenes = sorted(script["scenes"], key=lambda s: s.get("order", 999999))
playlists = script["playlist"]

print("JOB_ID:", JOB_ID)
print("Scenes:", len(scenes))
print("Playlists:", [p["clip_id"] for p in playlists])
print("Output:", OUTPUT_ROOT)

IMAGE_EXTS = [".png", ".jpg", ".jpeg", ".webp"]
VIDEO_EXTS = [".mp4", ".mov", ".mkv", ".webm"]

def candidate_video_paths(scene):
    scene_id = scene["scene_id"]
    scene_type = scene.get("scene_type", "")
    candidates = []

    for ext in VIDEO_EXTS:
        candidates.append(VIDEO_TEMPLATE_DIR / f"{scene_id}{ext}")

    if scene_type:
        for ext in VIDEO_EXTS:
            candidates.append(VIDEO_TEMPLATE_DIR / f"{scene_type}{ext}")

    return candidates

def candidate_image_paths(scene):
    scene_id = scene["scene_id"]
    scene_type = scene.get("scene_type", "")
    candidates = []

    for ext in IMAGE_EXTS:
        candidates.append(SCENE_IMAGE_DIR / f"{scene_id}{ext}")

    if scene_type:
        for ext in IMAGE_EXTS:
            candidates.append(SCENE_IMAGE_DIR / f"{scene_type}{ext}")

    return candidates

def find_visual_asset(scene):
    needs_lipsync = bool(scene.get("needs_lipsync", False))
    scene_type = scene.get("scene_type", "")

    # 1. Ưu tiên video đúng scene_id hoặc scene_type
    for p in candidate_video_paths(scene):
        if p.exists():
            return {"path": p, "kind": "video"}

    # 2. Fallback cho mọi scene cần lip-sync:
    # dùng HOST_TALK.mp4 trước, nếu không có thì dùng HOOK.mp4
    if needs_lipsync:
        for name in ["HOST_TALK.mp4", "HOOK.mp4"]:
            p = VIDEO_TEMPLATE_DIR / name
            if p.exists():
                return {"path": p, "kind": "video"}

    # 3. Ưu tiên ảnh đúng scene_id hoặc scene_type
    for p in candidate_image_paths(scene):
        if p.exists():
            if needs_lipsync and not ALLOW_STATIC_IMAGE_FOR_LIPSYNC:
                continue
            return {"path": p, "kind": "image"}

    # 4. Fallback cho product / non-lipsync scene:
    # dùng model.png nếu có
    if (not needs_lipsync) or scene_type == "PRODUCT_CLOSEUP":
        p = SCENE_IMAGE_DIR / "model.png"
        if p.exists():
            return {"path": p, "kind": "image"}

    return None

def quick_file_ok(path: Path) -> bool:
    """Check nhẹ trước khi ffprobe được định nghĩa: file tồn tại và không rỗng."""
    try:
        return path.exists() and path.is_file() and path.stat().st_size > 1024
    except Exception:
        return False

def scene_output_path(scene_id: str) -> Path:
    return SCENE_OUT_DIR / f"{scene_id}.mp4"

def validate_assets(scenes):
    rows = []
    missing = []

    for scene in scenes:
        scene_id = scene["scene_id"]
        audio_path = AUDIO_DIR / f"{scene_id}.wav"
        existing_scene = scene_output_path(scene_id)
        existing_scene_ok = quick_file_ok(existing_scene)
        visual = find_visual_asset(scene)

        row = {
            "scene_id": scene_id,
            "order": scene.get("order"),
            "scene_type": scene.get("scene_type"),
            "needs_lipsync": scene.get("needs_lipsync"),
            "audio_path": str(audio_path),
            "audio_exists": audio_path.exists(),
            "visual_path": str(visual["path"]) if visual else "",
            "visual_kind": visual["kind"] if visual else "",
            "existing_scene_path": str(existing_scene),
            "existing_scene_ok": existing_scene_ok,
            "overlay_text": scene.get("overlay_text") or "",
        }
        rows.append(row)

        # Resume mode: nếu scene final đã render rồi thì không block vì thiếu asset/audio gốc nữa.
        # Trường hợp file output bị lỗi sẽ được kiểm tra kỹ bằng ffprobe ở cell generate.
        if not existing_scene_ok and (not audio_path.exists() or visual is None):
            missing.append(row)

    report_path = REPORT_DIR / "asset_report.csv"
    with open(report_path, "w", encoding="utf-8-sig", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=list(rows[0].keys()))
        writer.writeheader()
        writer.writerows(rows)

    return rows, missing, report_path

asset_rows, missing_assets, asset_report_path = validate_assets(scenes)

print("Asset report:", asset_report_path)
print("Missing count:", len(missing_assets))

if missing_assets:
    print("\nMissing / blocked assets:")
    for r in missing_assets:
        print(
            f"- {r['scene_id']} | audio={r['audio_exists']} | "
            f"visual={bool(r['visual_path'])} | type={r['scene_type']} | "
            f"needs_lipsync={r['needs_lipsync']}"
        )
    raise RuntimeError("Thiếu asset. Upload đủ audio/video trước khi chạy tiếp.")
else:
    print("All assets are ready.")

## 5. Helper functions — FFmpeg, audio check, Wav2Lip local runtime

In [ ]:
def run(cmd, cwd=None, check=True, tail_chars=5000):
    """Run command with visible logs and captured output."""
    cmd = [str(x) for x in cmd]
    print("\n$ " + " ".join(cmd))
    result = subprocess.run(
        cmd,
        cwd=str(cwd) if cwd else None,
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT
    )
    if result.stdout:
        print(result.stdout[-tail_chars:])
    if check and result.returncode != 0:
        raise RuntimeError(
            f"Command failed with code {result.returncode}: {' '.join(cmd)}\n"
            f"Last output:\n{result.stdout[-tail_chars:]}"
        )
    return result

def ffprobe_duration(path: Path) -> float:
    result = subprocess.check_output([
        "ffprobe", "-v", "error",
        "-show_entries", "format=duration",
        "-of", "default=noprint_wrappers=1:nokey=1",
        str(path)
    ])
    return float(result.decode("utf-8").strip())

def has_audio(path: Path) -> bool:
    result = subprocess.check_output([
        "ffprobe", "-v", "error",
        "-select_streams", "a",
        "-show_entries", "stream=codec_type",
        "-of", "csv=p=0",
        str(path)
    ])
    return bool(result.decode("utf-8").strip())

def normalize_video_to_audio(template_video: Path, audio_path: Path, out_path: Path):
    """Loop/trim video theo audio duration, chuẩn hóa 9:16 1080x1920, 25fps, no audio."""
    dur = ffprobe_duration(audio_path)

    vf = (
        f"scale={TARGET_W}:{TARGET_H}:force_original_aspect_ratio=increase,"
        f"crop={TARGET_W}:{TARGET_H},"
        f"fps={FPS},"
        "format=yuv420p"
    )

    run([
        "ffmpeg", "-y",
        "-stream_loop", "-1",
        "-i", template_video,
        "-t", f"{dur:.3f}",
        "-an",
        "-vf", vf,
        "-c:v", "libx264",
        "-preset", PRESET,
        "-crf", str(CRF),
        "-pix_fmt", "yuv420p",
        out_path
    ])

    return out_path

def image_to_motion_video(image_path: Path, audio_path: Path, out_path: Path):
    """Tạo pseudo-video từ ảnh theo audio duration bằng zoompan nhẹ."""
    dur = ffprobe_duration(audio_path)
    frames = max(1, int(dur * FPS))

    vf = (
        f"scale={TARGET_W}:{TARGET_H}:force_original_aspect_ratio=increase,"
        f"crop={TARGET_W}:{TARGET_H},"
        f"zoompan=z='min(zoom+0.0008,1.06)':d={frames}:s={TARGET_W}x{TARGET_H}:fps={FPS},"
        "format=yuv420p"
    )

    run([
        "ffmpeg", "-y",
        "-loop", "1",
        "-i", image_path,
        "-t", f"{dur:.3f}",
        "-vf", vf,
        "-c:v", "libx264",
        "-preset", PRESET,
        "-crf", str(CRF),
        "-pix_fmt", "yuv420p",
        out_path
    ])

    return out_path

def mux_audio(video_path: Path, audio_path: Path, out_path: Path):
    """Gắn audio vào video, trim theo track ngắn hơn."""
    run([
        "ffmpeg", "-y",
        "-i", video_path,
        "-i", audio_path,
        "-map", "0:v:0",
        "-map", "1:a:0",
        "-c:v", "libx264",
        "-preset", PRESET,
        "-crf", str(CRF),
        "-c:a", "aac",
        "-ar", "44100",
        "-b:a", "192k",
        "-shortest",
        "-pix_fmt", "yuv420p",
        out_path
    ])
    return out_path

def ffmpeg_filter_escape_path(path: Path | str) -> str:
    """
    Escape path cho FFmpeg filtergraph.
    Dùng cho fontfile/textfile trong drawtext.
    """
    s = str(path)
    return s.replace("\\", "\\\\").replace(":", "\\:").replace("'", "\\'")

def write_overlay_textfile(out_path: Path, text: str) -> Path:
    """Ghi text overlay UTF-8 ra /content để FFmpeg đọc ổn định, tránh lỗi Unicode trên Drive."""
    overlay_runtime_dir = Path("/content/overlay_texts")
    overlay_runtime_dir.mkdir(parents=True, exist_ok=True)

    text_file = overlay_runtime_dir / f"{out_path.stem}_overlay.txt"
    text = str(text).replace("\r\n", "\n").replace("\r", "\n").strip()
    text_file.write_text(text, encoding="utf-8", newline="\n")
    return text_file

def is_final_text_scene(scene: dict | None) -> bool:
    """Detect scene cuối / CTA / outro để đưa text ra giữa màn hình."""
    if not scene:
        return False

    scene_id = str(scene.get("scene_id", ""))
    scene_type = str(scene.get("scene_type", "")).upper()
    role = str(scene.get("role", "")).upper()
    title = str(scene.get("title", "")).upper()

    final_types = {
        "CTA", "FINAL", "OUTRO", "END", "END_CARD", "FINAL_SCENE",
        "CLOSING", "CLOSE", "SUMMARY", "THANK_YOU"
    }

    if scene_type in final_types or role in final_types or "FINAL" in title or "CTA" in title:
        return True

    # Scene cuối trong playlist cuối cũng được xem là final/end-card.
    try:
        last_playlist = playlists[-1]
        last_scene_id = last_playlist.get("scenes", [])[-1]
        if scene_id == last_scene_id:
            return True
    except Exception:
        pass

    return False

def build_bottom_text_filters(text_file: Path) -> list[str]:
    """Overlay thường: caption dưới, có box nền chống lỗi đọc."""
    font = ffmpeg_filter_escape_path(FONT_PATH)
    textfile = ffmpeg_filter_escape_path(text_file)

    return [
        (
            f"drawtext=fontfile='{font}':"
            f"textfile='{textfile}':"
            "fontcolor=white:"
            "fontsize=42:"
            "line_spacing=10:"
            "box=1:"
            "boxcolor=black@0.62:"
            "boxborderw=24:"
            "shadowcolor=black@0.55:"
            "shadowx=2:"
            "shadowy=2:"
            "fix_bounds=1:"
            "x=(w-text_w)/2:"
            "y=h-text_h-180"
        )
    ]

def build_center_final_text_filters(text_file: Path) -> list[str]:
    """
    Final scene: text ở giữa + panel + line trang trí.
    Chỉ dùng FFmpeg filter cơ học, không cần plugin ngoài.
    """
    font = ffmpeg_filter_escape_path(FONT_PATH)
    textfile = ffmpeg_filter_escape_path(text_file)

    panel_h = 560
    panel_y = f"(h-{panel_h})/2"

    return [
        # Làm tối nhẹ để chữ nổi hơn.
        "eq=contrast=1.04:brightness=-0.035:saturation=1.05",

        # Panel trung tâm.
        f"drawbox=x=78:y={panel_y}:w=w-156:h={panel_h}:color=black@0.46:t=fill",

        # Viền / thanh trang trí kiểu edit cơ học.
        f"drawbox=x=110:y={panel_y}+28:w=w-220:h=4:color=white@0.82:t=fill",
        f"drawbox=x=110:y={panel_y}+{panel_h}-32:w=w-220:h=4:color=white@0.82:t=fill",
        f"drawbox=x=126:y={panel_y}+54:w=96:h=6:color=white@0.95:t=fill",
        f"drawbox=x=w-222:y={panel_y}+{panel_h}-68:w=96:h=6:color=white@0.95:t=fill",

        # Một scanline nháy nhẹ trong 0.1s mỗi 1.4s để tạo cảm giác edit cơ học.
        "drawbox=x=120:y=h/2-310:w=w-240:h=3:color=white@0.35:t=fill:enable='lt(mod(t,1.4),0.10)'",

        # Text chính ở giữa, dùng textfile UTF-8 để giữ dấu tiếng Việt.
        (
            f"drawtext=fontfile='{font}':"
            f"textfile='{textfile}':"
            "fontcolor=white:"
            "fontsize=56:"
            "line_spacing=18:"
            "box=0:"
            "shadowcolor=black@0.78:"
            "shadowx=3:"
            "shadowy=3:"
            "fix_bounds=1:"
            "x=(w-text_w)/2:"
            "y=(h-text_h)/2"
        ),
    ]

def apply_mechanical_edit_effects(video_path: Path, out_path: Path, extra_filters: list[str] | None = None):
    """
    Apply hiệu ứng edit cơ học nhẹ bằng FFmpeg:
    - fade in/out rất ngắn
    - filter overlay/text nếu có
    Giữ audio gốc.
    """
    extra_filters = extra_filters or []

    if not ENABLE_MECHANICAL_EDIT_EFFECTS and not extra_filters:
        shutil.copy2(video_path, out_path)
        return out_path

    dur = ffprobe_duration(video_path)
    fade_out_start = max(0.0, dur - 0.22)

    filters = [
        "format=yuv420p",
        "fade=t=in:st=0:d=0.12",
        f"fade=t=out:st={fade_out_start:.3f}:d=0.22",
    ]
    filters.extend(extra_filters)

    vf = ",".join(filters)

    run([
        "ffmpeg", "-y",
        "-i", video_path,
        "-vf", vf,
        "-map", "0:v:0",
        "-map", "0:a:0?",
        "-c:v", "libx264",
        "-preset", PRESET,
        "-crf", str(CRF),
        "-c:a", "aac",
        "-ar", "44100",
        "-b:a", "192k",
        "-pix_fmt", "yuv420p",
        out_path
    ])

    return out_path

def add_overlay_text(video_path: Path, text: str, out_path: Path, scene: dict | None = None):
    """
    Burn overlay text tiếng Việt bằng Noto Sans.
    - Dùng textfile UTF-8 local để tránh lỗi encoding/path trên Drive.
    - Scene cuối/CTA: text đặt giữa màn hình + panel trang trí.
    - Scene thường: caption dưới.
    """
    text = "" if text is None else str(text).replace("\r\n", "\n").replace("\r", "\n").strip()

    extra_filters = []
    if text:
        text_file = write_overlay_textfile(out_path, text)

        if is_final_text_scene(scene):
            extra_filters = build_center_final_text_filters(text_file)
        else:
            extra_filters = build_bottom_text_filters(text_file)

    # Dù không có text vẫn có thể apply fade cơ học nhẹ nếu ENABLE_MECHANICAL_EDIT_EFFECTS=True.
    apply_mechanical_edit_effects(video_path, out_path, extra_filters=extra_filters)

    return out_path

def run_wav2lip(face_video: Path, audio_path: Path, out_path: Path, scene_id: str):
    """
    Chạy Wav2Lip bắt buộc để sync lip.
    Copy input từ Drive sang /content để tránh lỗi I/O chậm.
    Sau Wav2Lip, ép mux lại audio gốc để chắc chắn file cuối có tiếng.
    """
    RUNTIME_DIR.mkdir(parents=True, exist_ok=True)

    local_face = RUNTIME_DIR / f"{scene_id}_face.mp4"
    local_audio = RUNTIME_DIR / f"{scene_id}_audio.wav"
    local_temp = RUNTIME_DIR / f"{scene_id}_wav2lip_temp.mp4"

    shutil.copy2(face_video, local_face)
    shutil.copy2(audio_path, local_audio)

    result_path = WAV2LIP_DIR / "results" / "result_voice.mp4"
    if result_path.exists():
        result_path.unlink()

    run([
        "python", "inference.py",
        "--checkpoint_path", CHECKPOINT_DST,
        "--face", local_face,
        "--audio", local_audio,
        "--pads", *PADS,
        "--resize_factor", RESIZE_FACTOR,
        "--wav2lip_batch_size", WAV2LIP_BATCH_SIZE,
    ], cwd=WAV2LIP_DIR)

    if not result_path.exists():
        raise RuntimeError(f"Wav2Lip không tạo result_voice.mp4 cho {scene_id}")

    shutil.copy2(result_path, local_temp)

    # Ép mux lại audio gốc để output chắc chắn có tiếng
    mux_audio(local_temp, local_audio, out_path)

    return out_path

## 6. Test nhanh Wav2Lip với scene đầu tiên cần lip-sync

In [ ]:
# Cell này test 1 scene lipsync trước để bắt lỗi sớm.
# Resume mode: mặc định bỏ qua để không chạy lại Wav2Lip không cần thiết.
first_lipsync_scene = next((s for s in scenes if bool(s.get("needs_lipsync", False))), None)

if not RUN_WAV2LIP_SMOKE_TEST:
    print("RUN_WAV2LIP_SMOKE_TEST=False -> bỏ qua test Wav2Lip để resume nhanh.")
elif first_lipsync_scene is None:
    print("Không có scene nào needs_lipsync=True. Bỏ qua test Wav2Lip.")
else:
    scene_id = first_lipsync_scene["scene_id"]
    audio_path = AUDIO_DIR / f"{scene_id}.wav"
    visual = find_visual_asset(first_lipsync_scene)

    test_base = WORK_DIR / f"{scene_id}_test_base.mp4"
    test_raw = WORK_DIR / f"{scene_id}_test_wav2lip.mp4"

    print("Test scene:", scene_id)
    print("Audio:", audio_path)
    print("Visual:", visual)

    if visual["kind"] == "video":
        normalize_video_to_audio(visual["path"], audio_path, test_base)
    else:
        image_to_motion_video(visual["path"], audio_path, test_base)

    run_wav2lip(test_base, audio_path, test_raw, scene_id + "_test")

    print("Wav2Lip test OK:", test_raw)
    print("Has audio:", has_audio(test_raw))

## 7. Generate từng scene — resume scene còn thiếu, bắt buộc lip-sync cho scene host

In [ ]:
generated_scenes = {}
scene_logs = []
scene_errors = []

def existing_scene_is_usable(path: Path) -> bool:
    """Check scene đã render có thể dùng lại: tồn tại, có audio, duration > 0."""
    if not quick_file_ok(path):
        return False

    try:
        return has_audio(path) and ffprobe_duration(path) > 0.1
    except Exception as e:
        print(f"Existing scene bị lỗi, sẽ render lại: {path} | {repr(e)}")
        return False

for scene in scenes:
    scene_id = scene["scene_id"]
    scene_type = scene.get("scene_type", "")
    needs_lipsync = bool(scene.get("needs_lipsync", False))
    overlay_text = scene.get("overlay_text")

    final_scene = SCENE_OUT_DIR / f"{scene_id}.mp4"

    print("\n" + "=" * 90)
    print(f"SCENE {scene_id} | type={scene_type} | lipsync={needs_lipsync}")
    print("=" * 90)

    # Resume: scene đã có output hợp lệ thì không render lại.
    if (
        RENDER_ONLY_MISSING_SCENES
        and not FORCE_REGENERATE_SCENES
        and existing_scene_is_usable(final_scene)
    ):
        generated_scenes[scene_id] = final_scene
        scene_logs.append({
            "scene_id": scene_id,
            "scene_type": scene_type,
            "needs_lipsync": needs_lipsync,
            "audio_path": "",
            "visual_path": "",
            "visual_kind": "",
            "output_path": str(final_scene),
            "status": "skipped_existing_ok"
        })
        print("SKIP existing scene OK:", final_scene)
        continue

    audio_path = AUDIO_DIR / f"{scene_id}.wav"
    visual = find_visual_asset(scene)

    if not audio_path.exists():
        msg = f"Missing audio: {audio_path}"
        print("SKIP:", msg)
        scene_errors.append({"scene_id": scene_id, "error": msg})
        if STRICT_LIPSYNC and needs_lipsync:
            raise RuntimeError(msg)
        continue

    if visual is None:
        msg = (
            f"Missing visual asset for {scene_id}. "
            f"Need video_templates/{scene_id}.mp4 or video_templates/{scene_type}.mp4. "
            f"For non-lipsync scenes, image fallback is scene_images/{scene_id}.png/jpg/webp."
        )
        print("SKIP:", msg)
        scene_errors.append({"scene_id": scene_id, "error": msg})
        if STRICT_LIPSYNC and needs_lipsync:
            raise RuntimeError(msg)
        continue

    try:
        asset_path = visual["path"]
        asset_kind = visual["kind"]

        base_video = WORK_DIR / f"{scene_id}_base.mp4"
        raw_scene = WORK_DIR / f"{scene_id}_raw.mp4"

        # Nếu file cũ lỗi, xóa trước khi render lại để tránh concat nhầm.
        if final_scene.exists():
            try:
                final_scene.unlink()
            except Exception:
                pass

        if asset_kind == "video":
            normalize_video_to_audio(asset_path, audio_path, base_video)
        elif asset_kind == "image":
            image_to_motion_video(asset_path, audio_path, base_video)
        else:
            raise ValueError(f"Unsupported visual kind: {asset_kind}")

        if needs_lipsync:
            # Không fallback mux-only. Nếu Wav2Lip lỗi thì fail để tránh lệch miệng.
            run_wav2lip(base_video, audio_path, raw_scene, scene_id)
        else:
            mux_audio(base_video, audio_path, raw_scene)

        add_overlay_text(raw_scene, overlay_text, final_scene, scene=scene)

        if not has_audio(final_scene):
            raise RuntimeError(f"Scene output không có audio: {final_scene}")

        generated_scenes[scene_id] = final_scene

        scene_logs.append({
            "scene_id": scene_id,
            "scene_type": scene_type,
            "needs_lipsync": needs_lipsync,
            "audio_path": str(audio_path),
            "visual_path": str(asset_path),
            "visual_kind": asset_kind,
            "output_path": str(final_scene),
            "status": "rendered_ok"
        })

        print("DONE:", final_scene)

    except Exception as e:
        err = repr(e)
        print("ERROR:", err)
        scene_errors.append({"scene_id": scene_id, "error": err})

        # Với scene lipsync, không tạo final lệch miệng.
        if STRICT_LIPSYNC and needs_lipsync:
            raise

# Save logs
manifest_path = REPORT_DIR / "generated_scene_manifest.csv"
if scene_logs:
    with open(manifest_path, "w", encoding="utf-8-sig", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=list(scene_logs[0].keys()))
        writer.writeheader()
        writer.writerows(scene_logs)

errors_path = REPORT_DIR / "scene_errors.json"
with open(errors_path, "w", encoding="utf-8") as f:
    json.dump(scene_errors, f, ensure_ascii=False, indent=2)

rendered_count = sum(1 for r in scene_logs if r.get("status") == "rendered_ok")
skipped_count = sum(1 for r in scene_logs if r.get("status") == "skipped_existing_ok")

print("\nUsable scenes:", len(generated_scenes))
print("Rendered new scenes:", rendered_count)
print("Skipped existing scenes:", skipped_count)
print("Scene manifest:", manifest_path)
print("Errors:", errors_path)

if scene_errors:
    print("Scene errors:")
    print(json.dumps(scene_errors[:10], ensure_ascii=False, indent=2))

## 8. Concat playlist thành loop + final video

In [ ]:
def concat_videos(video_paths, out_path: Path):
    """Concat bằng demuxer + re-encode để tránh mismatch codec/timebase/audio."""
    if not video_paths:
        raise ValueError(f"No videos to concat for {out_path}")

    concat_file = WORK_DIR / f"{out_path.stem}_concat.txt"

    lines = []
    for p in video_paths:
        p = Path(p).resolve()
        lines.append(f"file '{p}'")

    concat_file.write_text("\n".join(lines), encoding="utf-8")

    run([
        "ffmpeg", "-y",
        "-f", "concat",
        "-safe", "0",
        "-i", concat_file,
        "-c:v", "libx264",
        "-preset", PRESET,
        "-crf", str(CRF),
        "-c:a", "aac",
        "-ar", "44100",
        "-b:a", "192k",
        "-pix_fmt", "yuv420p",
        out_path
    ])

    if not has_audio(out_path):
        raise RuntimeError(f"Concat output không có audio: {out_path}")

    return out_path

loop_outputs = []

for playlist in playlists:
    clip_id = playlist["clip_id"]
    scene_ids = playlist.get("scenes", [])

    paths = []
    missing = []

    for sid in scene_ids:
        scene_path = SCENE_OUT_DIR / f"{sid}.mp4"
        if existing_scene_is_usable(scene_path):
            paths.append(scene_path)
        else:
            missing.append(sid)

    print("\nPlaylist:", clip_id)
    print("Included:", [Path(p).name for p in paths])
    if missing:
        print("Missing / unusable scenes:", missing)

    if missing:
        raise RuntimeError(f"Playlist {clip_id} thiếu scene hoặc scene lỗi: {missing}. Không tạo final thiếu đoạn.")

    out_path = FINAL_OUT_DIR / f"{clip_id}.mp4"

    if out_path.exists() and not REBUILD_FINAL_VIDEO and existing_scene_is_usable(out_path):
        print("SKIP existing playlist final OK:", out_path)
    else:
        concat_videos(paths, out_path)
        print("LOOP DONE:", out_path)

    loop_outputs.append(out_path)

if loop_outputs:
    full_video = FINAL_OUT_DIR / f"{JOB_ID}_FULL_LOOP.mp4"

    if full_video.exists() and not REBUILD_FINAL_VIDEO and existing_scene_is_usable(full_video):
        print("SKIP existing FULL LOOP OK:", full_video)
    else:
        concat_videos(loop_outputs, full_video)
        print("\nFINAL VIDEO:", full_video)

    print("HAS AUDIO:", has_audio(full_video))
else:
    full_video = None
    raise RuntimeError("No loop outputs generated.")

## 9. Preview kết quả trong Colab

In [ ]:
from IPython.display import Video, display

if full_video and Path(full_video).exists():
    display(Video(str(full_video), embed=True, width=360))
else:
    print("Chưa có full video để preview.")

## 10. Zip output để tải về

In [ ]:
zip_base = DATA_DIR / "outputs" / f"{JOB_ID}_video_package"
zip_path = shutil.make_archive(
    base_name=str(zip_base),
    format="zip",
    root_dir=str(OUTPUT_ROOT)
)

print("ZIP:", zip_path)

try:
    from google.colab import files
    files.download(zip_path)
except Exception as e:
    print("Không auto-download được, tải thủ công tại:", zip_path)
    print("Error:", repr(e))

## 11. Backend manifest

In [ ]:
backend_manifest = {
    "job_id": JOB_ID,
    "created_at": datetime.now(UTC).isoformat().replace("+00:00", "Z"),
    "output_root": str(OUTPUT_ROOT),
    "scene_dir": str(SCENE_OUT_DIR),
    "final_dir": str(FINAL_OUT_DIR),
    "full_video": str(full_video) if full_video else None,
    "loops": [str(p) for p in loop_outputs],
    "reports": {
        "asset_report": str(asset_report_path),
        "scene_manifest": str(REPORT_DIR / "generated_scene_manifest.csv"),
        "errors": str(REPORT_DIR / "scene_errors.json"),
    },
    "lipsync": {
        "strict_lipsync": STRICT_LIPSYNC,
        "resize_factor": RESIZE_FACTOR,
        "pads": PADS,
        "checkpoint": str(CHECKPOINT_DST),
        "runtime_dir": str(RUNTIME_DIR),
    }
}

manifest_json_path = OUTPUT_ROOT / "video_pipeline_manifest.json"
with open(manifest_json_path, "w", encoding="utf-8") as f:
    json.dump(backend_manifest, f, ensure_ascii=False, indent=2)

print("Backend manifest:", manifest_json_path)
backend_manifest

## Ghi chú input visual

Cách tốt nhất là dùng video riêng từng scene:

```text
data/video_templates/S001.mp4
data/video_templates/S002.mp4
...
```

Nếu chưa có, dùng fallback theo `scene_type`:

```text
data/video_templates/HOST_TALK.mp4
data/video_templates/CTA.mp4
data/video_templates/HOST_PHONE_READING.mp4
data/video_templates/FAQ_ANSWER.mp4
data/video_templates/PRODUCT_CLOSEUP.mp4
```

Với scene `needs_lipsync=True`, video mẫu nên là:
- mặt rõ, nhìn thẳng hoặc gần thẳng
- miệng không bị che
- chuyển động đầu nhẹ
- mặt đủ lớn trong khung hình
- không dùng video mẫu đang nói quá mạnh nếu muốn kết quả tự nhiên hơn

Với scene `PRODUCT_CLOSEUP` / `needs_lipsync=False`, có thể dùng ảnh:

```text
data/scene_images/S003.png
data/scene_images/S006.png
...
```